In [1]:
import pandas as pd

# Load your data (adjust paths if needed)
train = pd.read_csv('train.csv')  # your main train file

# Sort by time (important for sequences)
train = train.sort_values(['date_id', 'time_id']).reset_index(drop=True)

# Define columns (match what you have)
FEATURE_COLS = [f'f{i}' for i in range(26)]
TARGET_COL   = 'y'
SYMBOL_COL   = 'symbol_id'

# Optional: Handle missing values (e.g., f1 has NaNs)
for col in FEATURE_COLS:
    if train[col].isna().any():
        train[col] = train[col].fillna(train[col].median())

# Time-respecting split (80/20 by date_id)
split_date = train['date_id'].quantile(0.8)
train_df = train[train['date_id'] <= split_date].copy()
valid_df = train[train['date_id'] > split_date].copy()

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Ready for dataset creation!")

Train shape: (1614334, 31)
Valid shape: (402793, 31)
Ready for dataset creation!


In [2]:
if 'train_df' not in globals() or 'valid_df' not in globals():
    raise ValueError("train_df or valid_df not defined! Run the data loading cell first.")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os

# Set CPU threads to match your 4 cores
torch.set_num_threads(4)
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
print("PyTorch threads set to 4 (for your 4-core CPU)")

# Your column names (copy from what you have)
FEATURE_COLS = [f"f{i}" for i in range(26)]
TARGET_COL = "y"
SYMBOL_COL = "symbol_id"

# Dataset (same as before)
class SimpleSequenceDataset(Dataset):
    def __init__(self, df, feature_cols, target_col, symbol_col=SYMBOL_COL, seq_len=10):
        self.groups = df.groupby(symbol_col)
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.seq_len = seq_len
        self.data = []
        for _, group in self.groups:
            X = group[feature_cols].values.astype(np.float32)
            y = group[target_col].values.astype(np.float32)
            for i in range(len(group) - seq_len):
                self.data.append((X[i:i+seq_len], y[i+seq_len]))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        X_seq, y = self.data[idx]
        return torch.tensor(X_seq), torch.tensor(y)

# GRU model with smoothness
class StateSpaceGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim + input_dim, 1)
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        gru_out, hn = self.gru(x)
        hn = hn.squeeze(0)
        last_x = x[:, -1, :]
        pred = self.fc(self.dropout(torch.cat([last_x, hn], dim=1))).squeeze()
        
        # Smoothness loss (encourage gradual hidden state changes)
        if x.size(1) > 1:
            state_diff = (gru_out[:, 1:] - gru_out[:, :-1]).pow(2).mean()
        else:
            state_diff = torch.tensor(0.0, device=x.device)
        
        return pred, state_diff

# ────────────────────────────────────────────────
# Setup & run
# ────────────────────────────────────────────────
seq_len = 10          # shorter = faster
batch_size = 1024

train_dataset = SimpleSequenceDataset(train_df, FEATURE_COLS, TARGET_COL, seq_len=seq_len)
valid_dataset = SimpleSequenceDataset(valid_df, FEATURE_COLS, TARGET_COL, seq_len=seq_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

device = torch.device('cpu')  # no GPU
model = StateSpaceGRU(len(FEATURE_COLS), hidden_dim=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
criterion = nn.MSELoss()

lambda_smooth = 0.05  # small penalty for persistent states

print("Starting training... Expect first epoch in 15–45 min on 4-core CPU")
for epoch in range(8):  # only 3 to start
    model.train()
    train_loss = 0
    batch_idx = 0
    total_batches = len(train_loader)
    
    for X_seq, y in train_loader:
        X_seq, y = X_seq.to(device), y.to(device)
        optimizer.zero_grad()
        pred, state_diff = model(X_seq)
        loss = criterion(pred, y) + lambda_smooth * state_diff
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(y)
        
        batch_idx += 1
        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{total_batches} | Loss: {loss.item():.6f}")
    
    train_rmse = np.sqrt(train_loss / len(train_loader.dataset))
    print(f"Epoch {epoch+1:2d} | Train RMSE: {train_rmse:.6f}")

# Quick validation RMSE
model.eval()
valid_preds = []
valid_ys = []
with torch.no_grad():
    for X_seq, y in valid_loader:
        X_seq = X_seq.to(device)
        pred, _ = model(X_seq)
        valid_preds.extend(pred.cpu().numpy())
        valid_ys.extend(y.numpy())

valid_rmse = np.sqrt(np.mean((np.array(valid_ys) - np.array(valid_preds))**2))
print(f"\nValid RMSE after 3 epochs: {valid_rmse:.6f}")
print("Compare to your LightGBM RMSE (~0.00225)")

PyTorch threads set to 4 (for your 4-core CPU)
Starting training... Expect first epoch in 15–45 min on 4-core CPU
Epoch 1 | Batch 100/1575 | Loss: 0.348313
Epoch 1 | Batch 200/1575 | Loss: 0.464653
Epoch 1 | Batch 300/1575 | Loss: 0.336904
Epoch 1 | Batch 400/1575 | Loss: 0.134210
Epoch 1 | Batch 500/1575 | Loss: 0.093718
Epoch 1 | Batch 600/1575 | Loss: 0.053073
Epoch 1 | Batch 700/1575 | Loss: 0.092328
Epoch 1 | Batch 800/1575 | Loss: 0.052517
Epoch 1 | Batch 900/1575 | Loss: 0.044557
Epoch 1 | Batch 1000/1575 | Loss: 0.027178
Epoch 1 | Batch 1100/1575 | Loss: 0.089697
Epoch 1 | Batch 1200/1575 | Loss: 0.015491
Epoch 1 | Batch 1300/1575 | Loss: 0.012233
Epoch 1 | Batch 1400/1575 | Loss: 0.011826
Epoch 1 | Batch 1500/1575 | Loss: 0.021584
Epoch  1 | Train RMSE: 0.459042
Epoch 2 | Batch 100/1575 | Loss: 0.011915
Epoch 2 | Batch 200/1575 | Loss: 0.007245
Epoch 2 | Batch 300/1575 | Loss: 0.007829
Epoch 2 | Batch 400/1575 | Loss: 0.005238
Epoch 2 | Batch 500/1575 | Loss: 0.004335
Epoch 2 